Step 1: Setting Up the Spark Environment In a real-world company setup, we wouldn't use Google Colab directly. Instead, we would:

1\. Deploy a Spark Cluster (like AWS EMR, GCP Dataproc, or an on-prem Hadoop cluster, Azure HD Insight).

2\. Store Data in HDFS instead of local storage. Load data from Kaggle i.e. Data Source

3\. (#!/bin/bash curl-L-o\~/Downloads/brazilian-ecommerce.zip) https\://www\.kaggle.com/api/v1/datasets/download/olistbr/brazilian- ecommerce)

4\. Use PySpark to interact with data.


In [ ]:
!pwd

/content


In [ ]:
#!/bin/bash
!curl -L -o /content/brazilian-ecommerce.zip\
  https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.6M  100 42.6M    0     0  55.8M      0 --:--:-- --:--:-- --:--:--  368M


In [ ]:
!unzip /content/brazilian-ecommerce.zip  -d /content/brazilian-e-commerce/

Archive:  /content/brazilian-ecommerce.zip
  inflating: /content/brazilian-e-commerce/olist_customers_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_geolocation_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_order_items_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_order_payments_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_order_reviews_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_orders_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_products_dataset.csv  
  inflating: /content/brazilian-e-commerce/olist_sellers_dataset.csv  
  inflating: /content/brazilian-e-commerce/product_category_name_translation.csv  


In [ ]:
! ls -lrt -h /content/brazilian-e-commerce/

total 121M
-rw-r--r-- 1 root root 8.7M Oct  1  2021 olist_customers_dataset.csv
-rw-r--r-- 1 root root  59M Oct  1  2021 olist_geolocation_dataset.csv
-rw-r--r-- 1 root root  15M Oct  1  2021 olist_order_items_dataset.csv
-rw-r--r-- 1 root root 5.6M Oct  1  2021 olist_order_payments_dataset.csv
-rw-r--r-- 1 root root  17M Oct  1  2021 olist_orders_dataset.csv
-rw-r--r-- 1 root root  14M Oct  1  2021 olist_order_reviews_dataset.csv
-rw-r--r-- 1 root root 2.3M Oct  1  2021 olist_products_dataset.csv
-rw-r--r-- 1 root root 2.6K Oct  1  2021 product_category_name_translation.csv
-rw-r--r-- 1 root root 171K Oct  1  2021 olist_sellers_dataset.csv


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()


In [ ]:
customer_df=spark.read.csv("/content/brazilian-e-commerce/olist_customers_dataset.csv",header=True,inferSchema=True)
print(customer_df.printSchema())
customer_df.show(10)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

None
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c8

In [ ]:
## Prompt: Do the previous cell step for all the files in the brazillian ecommerce folder and show it properly

In [ ]:
import os
directory_path = '/content/brazilian-e-commerce/'
all_files = os.listdir(directory_path)

csv_files = [file for file in all_files if file.endswith('.csv')]
print("CSV files in the directory:")
for csv_file in csv_files:
    print(csv_file)
    parts = csv_file.replace('.csv', '').split('_')
    if len(parts) == 3:
        name = parts[1]
    elif len(parts) == 4:
        name = f"{parts[1]}_{parts[2]}"
    else:
        continue  # skip unexpected patterns safely
    df_var_name = f"df_{name}"
    globals()[df_var_name] = spark.read.csv(
        f"{directory_path}{csv_file}",
        header=True,
        inferSchema=True
    )

CSV files in the directory:
olist_orders_dataset.csv
olist_sellers_dataset.csv
olist_order_payments_dataset.csv
olist_products_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
product_category_name_translation.csv
olist_order_reviews_dataset.csv
olist_order_items_dataset.csv


In [ ]:
df_orders.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [ ]:
df_category_name.show(10)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
|        esporte_lazer|               sports_leisure|
|           perfumaria|                    perfumery|
| utilidades_domest...|                   housewares|
|            telefonia|                    telephony|
|   relogios_presentes|                watches_gifts|
+---------------------+-----------------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import count,col,when
customer_df.select([count(when(col(c).isNull(),c)).alias(c) for c in customer_df.columns]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [ ]:
customer_df.groupBy('customer_id').count().filter('count>1').show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [ ]:
customer_df.groupBy('customer_state').count().orderBy('count',ascending=False).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows


In [ ]:
df_orders.count()

99441

In [ ]:
df_orders.groupBy('customer_id','order_status').count().filter('order_status=="delivered"').show()

+--------------------+------------+-----+
|         customer_id|order_status|count|
+--------------------+------------+-----+
|295ae9b35379e0772...|   delivered|    1|
|5f2bc926ec92aa65c...|   delivered|    1|
|7bb3b0d45b4e1a13c...|   delivered|    1|
|9ce1401cf82bd5420...|   delivered|    1|
|06872c61a13a4d027...|   delivered|    1|
|c495ae8dfa83b158b...|   delivered|    1|
|4d69348be3bb86e33...|   delivered|    1|
|f59c3b73fe7833bf5...|   delivered|    1|
|c47f0ed7fbd21fc9b...|   delivered|    1|
|05ad208d1e3ae7d49...|   delivered|    1|
|d32393bafaf1d7caa...|   delivered|    1|
|4c756c39c13545719...|   delivered|    1|
|edd33f337f01e490c...|   delivered|    1|
|154e666b681104319...|   delivered|    1|
|b7aba705e32253c6d...|   delivered|    1|
|c5293253d2861ddfc...|   delivered|    1|
|4c6cc85a89f512df7...|   delivered|    1|
|d5c3f175135abfeee...|   delivered|    1|
|f63db8b1515a5d360...|   delivered|    1|
|da90cf906fefe4363...|   delivered|    1|
+--------------------+------------

In [ ]:
df_orders.groupBy('order_status').count().orderBy('count',ascending=False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [ ]:
# Top Selling Products

In [ ]:
df_order_items.show(10)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [ ]:
from pyspark.sql.functions import sum
df_order_items.groupBy('product_id').agg(sum('price').alias('total_price')).orderBy('total_price',ascending=False).show()

+--------------------+------------------+
|          product_id|       total_price|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
|e0d64dcfaa3b6db5c...|          31786.82|
|d285360f29ac7fd97...|31623.809999999983|
|7a10781637204d8d1...|           30467.5|
|f1c7f353075ce59d8...|          29997.36|
|f819f0c84a64f02d3...|29024.479999999996|
|588531f8ec37e7d5f...|28291.989999999998|
|422879e10f4668299...|26577.219999999972|
|16c4e87b98a9370a9...|           25034.0|
|5a848e4ab52fd5445...|24229.029999999962|
|a62e25e09e05e6faf...|           24051.0|
+--------------------+------------

In [ ]:
df_orders.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [ ]:
delivery_df=df_orders.select('customer_id','order_purchase_timestamp','order_delivered_customer_date')

In [ ]:
delivery_df.show(5)

+--------------------+------------------------+-----------------------------+
|         customer_id|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------------------+-----------------------------+
|9ef432eb625129730...|     2017-10-02 10:56:33|          2017-10-10 21:25:13|
|b0830fb4747a6c6d2...|     2018-07-24 20:41:37|          2018-08-07 15:27:45|
|41ce2a54c0b03bf34...|     2018-08-08 08:38:49|          2018-08-17 18:06:29|
|f88197465ea7920ad...|     2017-11-18 19:28:06|          2017-12-02 00:28:42|
|8ab97904e6daea886...|     2018-02-13 21:18:39|          2018-02-16 18:17:02|
+--------------------+------------------------+-----------------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import date_diff,to_date
delivery_detail_df=delivery_df.withColumn('delivery_days',date_diff(to_date('order_delivered_customer_date'),to_date('order_purchase_timestamp')))

In [ ]:
delivery_detail_df.orderBy('delivery_days',ascending=False).show(5)

+--------------------+------------------------+-----------------------------+-------------+
|         customer_id|order_purchase_timestamp|order_delivered_customer_date|delivery_days|
+--------------------+------------------------+-----------------------------+-------------+
|75683a92331068e2d...|     2017-02-21 23:31:27|          2017-09-19 14:36:39|          210|
|d306426abe5fca15e...|     2018-02-23 14:57:35|          2018-09-19 23:24:07|          208|
|7815125148cfa1e8c...|     2017-03-07 23:59:51|          2017-09-19 15:12:50|          196|
|217906bc11a32c1e4...|     2017-03-08 18:09:02|          2017-09-19 14:33:17|          195|
|9cf2c3fa2632cee74...|     2017-03-08 22:47:40|          2017-09-19 14:00:04|          195|
+--------------------+------------------------+-----------------------------+-------------+
only showing top 5 rows


In [ ]:
dfs = {
    'df_orders': df_orders,
    'df_order_reviews': df_order_reviews,
    'df_sellers': df_sellers,
    'df_category_name': df_category_name,
    'df_order_payments': df_order_payments,
    'df_order_items': df_order_items,
    'df_geolocation': df_geolocation,
    'df_customers': df_customers,
    'df_products': df_products
}


## We are checking how many missing values are there in each dataFrame

In [ ]:
def missing_values(dfs):
  for key,values in dfs.items():
    print(f"missing values of {key}")
    print(f"Count of all rows{values.count()}")
    values.select([count(when(col(c).isNull(),c)).alias(c) for c in values.columns]).show()

In [ ]:
missing_values(dfs)

missing values of df_orders
Count of all rows99441
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

missing values of df_order_reviews
Count of all rows104162
+---------+--------+------------+----------------

In [ ]:
df_orders.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [ ]:
orders_df_cleaned=df_orders.na.drop(subset=['order_id','customer_id','order_status'])

In [ ]:
orders_df_cleaned.count()

99441

In [ ]:
from pyspark.ml.feature import Imputer

In [ ]:
df_order_payments.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [ ]:
imputer=Imputer(inputCol='payment_value',outputCol='payment_value_imputed').setStrategy('mean')

In [ ]:
cleaned_payments_df=imputer.fit(df_order_payments).transform(df_order_payments)

In [ ]:
cleaned_payments_df.show(5)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|                99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|                24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|                65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|               107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|               128.45|
+--------------------+------------------+------------+--------------------+-------------+---------------

In [ ]:
payments_with_null=df_order_payments.filter(df_order_payments.payment_value.isNull())

In [ ]:
payments_with_null.show()

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
+--------+------------------+------------+--------------------+-------------+



In [ ]:
cleaned_payments_df.groupBy('payment_type').count().show()


+------------+-----+
|payment_type|count|
+------------+-----+
|      boleto|19784|
| not_defined|    3|
| credit_card|76795|
|     voucher| 5775|
|  debit_card| 1529|
+------------+-----+



In [ ]:
cleaned_payments_df=cleaned_payments_df.withColumn('payment_type',when(col('payment_type')=='boleto','Bank Transfer').when(col('payment_type')=='credit_card','Credit Card').when(col('payment_type')=='debit_card','Debit Card').otherwise('Other'))

In [ ]:
cleaned_payments_df.groupBy('payment_type').count().show()


+-------------+-----+
| payment_type|count|
+-------------+-----+
|  Credit Card|76795|
|Bank Transfer|19784|
|        Other| 5778|
|   Debit Card| 1529|
+-------------+-----+



In [ ]:
df_customers_cleaned=df_customers.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))

In [ ]:
df_customers_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [ ]:
df_customers_cleaned=df_customers_cleaned.dropDuplicates(['customer_id'])

In [ ]:
order_with_details=orders_df_cleaned.join(df_order_items,on='order_id',how='left')\
.join(cleaned_payments_df,on='order_id',how='left')\
.join(df_customers_cleaned,on='customer_id',how='left')

In [ ]:
order_with_details.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+------------------+-------------+--------------------+-------------+---------------------+--------------------+------------------------+--------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|payment_sequential| payment_type|payment_installments|payment_value|payment_value_imputed|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|
+--------------------+--------------------+------------+------------------------+-------------

In [ ]:
orders_with_totalValue=order_with_details.groupBy('order_id').agg(sum('price').alias('total_price'))

In [ ]:
orders_with_totalValue.show(10)

+--------------------+-----------+
|            order_id|total_price|
+--------------------+-----------+
|f373335aac9a659de...|       35.9|
|118045506e1c1dda0...|      225.0|
|cc66dee6fbc18bb79...|      117.7|
|f44cb69655f8e4d13...|     311.94|
|edcc6b79e8394346b...|       99.9|
|9f98d6530155e3b38...|      299.9|
|5e57ff5e1c008db89...|     139.98|
|0957ed870116e596b...|     129.99|
|3fa59277573f0fe06...|       79.0|
|d5f812041d8fc446c...|       64.9|
+--------------------+-----------+
only showing top 10 rows


## Advanced Transformation

In [ ]:
quantiles=df_order_items.approxQuantile('price',[0.01,0.99],0.00)
low,high=quantiles[0],quantiles[1]

In [ ]:
high,low

(890.0, 9.99)

In [ ]:
order_item_df_cleaned = df_order_items.filter((col('price') >=low) & (col('price') <= high))

In [ ]:
cleaned_payments_df.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|   2.853348863176944|
| stddev|  2.6870506738564925|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  24|
+-------+--------------------+



In [ ]:
products_df_cleaned = df_products.withColumn(
'produtt_size_category',
when(col('product_weight_g') <500,'Small')
.when(col('product_weight_g').between(500,2000),'Medium')
.otherwise('Large'))

In [ ]:
products_df_cleaned.groupBy('produtt_size_category').count().show()

+---------------------+-----+
|produtt_size_category|count|
+---------------------+-----+
|               Medium|12736|
|                Small|12464|
|                Large| 7751|
+---------------------+-----+



In [ ]:
# order_with_details.write.mode('overwrite').parquet('cleaned_data.parquet')

In [ ]:
# CREATE EXTERNAL TABLE cleaned_orders (
# product_id STRING,
# product_category_name STRING

# )
# STORED AS PARQUET
# LOCATION '/data/olist_proc/product_df_cleaned.parquet';|

In [ ]:
order_with_details.write.mode('overwrite').parquet('cleaned_data.parquet')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
for key,values in dfs.items():
  values.write.mode('overwrite').parquet(f'/content/drive/MyDrive/cleaned_data/{key}.parquet')

In [ ]:
cleaned_payments_df.write.mode('overwrite').parquet(f'/content/drive/MyDrive/cleaned_data/{'cleaned_payments_df'}.parquet')
df_customers_cleaned.write.mode('overwrite').parquet(f'/content/drive/MyDrive/cleaned_data/{'df_customers_cleaned'}.parquet')


In [73]:
spark.stop()